In [1]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')
final_recommendations = pd.read_parquet('recommendations.parquet')
als_recommendations = pd.read_parquet('als_recommendations.parquet')
items = pd.read_parquet('item_categories.parquet')

In [3]:
# NDCG@10

import numpy as np

def dcg_at_k(ranked_items, relevant_items, k):
    ranked_items = ranked_items[:k]
    dcg = 0.0
    for i, item in enumerate(ranked_items, start=1):
        rel = 1 if item in relevant_items else 0
        dcg += rel / np.log2(i + 1)
    return dcg

def ndcg_at_k(ranked_items, relevant_items, k):
    ideal_dcg = dcg_at_k(list(relevant_items), relevant_items, k)  # best possible ordering
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(ranked_items, relevant_items, k) / ideal_dcg

def mean_ndcg_at_k(user_rankings, user_relevant, k):
    scores = []
    for user_id, ranked_items in user_rankings.items():
        relevant_items = user_relevant.get(user_id, set())
        scores.append(ndcg_at_k(ranked_items, relevant_items, k))
    return np.mean(scores)

In [ ]:
# топ популярных
top_k_pop_items = events_train.groupby('item_id').size().reset_index(name='count').sort_values(['count'], ascending=False)[:10]

users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

cold_users = set(users_test) - set(users_train)

cold_users_events_with_recs = events_test[events_test["user_id"].isin(cold_users)].copy()
cold_users_events_with_recs["target"] = cold_users_events_with_recs["item_id"].isin(top_k_pop_items["item_id"]).astype(int)

precision = cold_users_events_with_recs.groupby("user_id")["target"].sum() / 100
precision_top = precision.mean()
print(f'TOP Precision: {precision_top:.6f}')

recall_top = cold_users_events_with_recs.groupby("user_id")["target"].mean().mean()
print(f'TOP Recall: {recall_top:.6f}')

cold_user_rankings = {user_id: top_k_pop_items['item_id'].tolist() for user_id in cold_users}
cold_user_relevant = (events_test[events_test['user_id'].isin(cold_users)]
                        .groupby('user_id')['item_id']
                        .apply(set)
                        .to_dict())

ndcg_pop = mean_ndcg_at_k(cold_user_rankings, cold_user_relevant, k=10)
print(f'TOP NDCG: {ndcg_pop:.6f}')


TOP Precision: 0.000084
TOP Recall: 0.004754
TOP NDCG: 0.002347
Coverage: 0.00


In [4]:
# персональные ALS

def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common, recs_for_common_users, events_for_common_users

In [5]:
def compute_cls_metrics(events_recs_for_binary_metrics):
    
    groupper = events_recs_for_binary_metrics.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fn"].sum())
    recall = recall.fillna(0).mean()

    return precision, recall

In [5]:
# als metrics

events_recs_for_binary_metrics, recs_for_common_users, events_for_common_users = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=10)

precision_als, recall_als = compute_cls_metrics(events_recs_for_binary_metrics)

#NDCG
user_rankings = (recs_for_common_users.groupby('user_id')['item_id']
                                        .apply(list)
                                        .to_dict())

user_relevant = (events_for_common_users.groupby('user_id')['item_id']
                                          .apply(set)
                                          .to_dict())

ndcg_als = mean_ndcg_at_k(user_rankings, user_relevant, k=10)

print(f'Precision: {precision_als:.6f}')
print(f'Recall: {recall_als:.6f}')
print(f'NDCG: {ndcg_als:.6f}')

Common users: 311128


/tmp/ipykernel_2268/332359964.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


Precision: 0.000602
Recall: 0.002361
NDCG: 0.001765


In [6]:
# итоговые

# для экономии ресурсов оставим события только тех пользователей, 
# для которых следует оценить рекомендации
events_inference = pd.concat([events_train, events_test])
events_inference = events_inference[events_inference["user_id"].isin(events_test["user_id"].drop_duplicates())]

cb_events_recs_for_binary_metrics_5, recs_for_common_users, events_for_common_users = process_events_recs_for_binary_metrics(
    events_inference,
    events_test,
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=10)

cb_precision_10, cb_recall_10 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

user_rankings = (recs_for_common_users.groupby('user_id')['item_id']
                                        .apply(list)
                                        .to_dict())

user_relevant = (events_for_common_users.groupby('user_id')['item_id']
                                          .apply(set)
                                          .to_dict())

ndcg_cb = mean_ndcg_at_k(user_rankings, user_relevant, k=10)

# novelty
# разметим каждую рекомендацию признаком played
events_train["played"] = True
final_recommendations = final_recommendations.merge(events_train, on=["user_id", "item_id"], how="left")
final_recommendations["played"] = final_recommendations["played"].fillna(False).astype("bool")

# проставим ранги
final_recommendations = final_recommendations.sort_values(by='cb_score', ascending=False)
final_recommendations["rank"] = final_recommendations.groupby("user_id").cumcount() + 1

# посчитаем novelty по пользователям
novelty_5_fin = (1-final_recommendations.query("rank <= 5").groupby("user_id")["played"].mean())

# coverage
items = pd.read_parquet('item_categories.parquet')

n_init = items['item_id'].nunique()
n_als = final_recommendations['item_id'].nunique()

cov_items_fin = n_als / n_init

print(f'Precision: {cb_precision_10:.6f}')
print(f'Recall: {cb_recall_10:.6f}')
print(f'NDCG: {ndcg_cb:.6f}')
print(f'Novelty: {novelty_5_fin.mean():.6f}')
print(f"Coverage: {cov_items_fin:.2f}")

Common users: 311128


/tmp/ipykernel_4195/332359964.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
/tmp/ipykernel_4195/1864791207.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  final_recommendations["played"] = final_recommendations["played"].fillna(False).astype("bool")


Precision: 0.001084
Recall: 0.004195
NDCG: 0.004433
Novelty: 0.996929
Coverage: 0.02
